# Setup

This notebook is made for a Linux distribution. Cmake and a c++ compiler are required, as well as Python.


To run, create a venv to run this notebook and install the following packages

In [8]:
%pip install pandas matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


Now we need to build using CMake. **First update your root directory**

In [10]:
import os, sys

ROOT_DIR = "/home/ayoub/Mr-SEQL"
os.chdir(ROOT_DIR)
sys.path.append(os.path.join(ROOT_DIR))

In [11]:
%cd {ROOT_DIR}
%mkdir -p build/Release
%cd build/Release
!cmake -DCMAKE_BUILD_TYPE=Release {ROOT_DIR}/src
!make
!cd bin

/home/ayoub/Mr-SEQL


/home/ayoub/Mr-SEQL/build/Release
CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.5 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value or use a ...<max> suffix to tell
  CMake that the project does not need compatibility with older versions.


-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/ayoub/Mr-SEQL/build/Release
[ 25%] Built target sax_convert
[ 75%] Built target mr_seql
[100%] Built target compute_metats


In [12]:
def prepare_data_sax(dataset_name, saxdir_name):
    %cd {ROOT_DIR}/build/Release/bin/
    path = os.path.join(ROOT_DIR, 'data', dataset_name)
    # Convert FordB format: remove leading spaces and replace multiple spaces with commas
    if dataset_name in  ['FordB', 'MelbournePedestrian']:
        !cp {path}/{dataset_name}_TRAIN.txt .
        !cp {path}/{dataset_name}_TEST.txt .
        import re
        for file_suffix in ['TRAIN', 'TEST']:
            filenametxt = f'{dataset_name}_{file_suffix}.txt'
            with open(filenametxt, 'r') as f:
                lines = f.readlines()
        
            # Process each line
            converted_lines = []
            for line in lines:
                # Remove leading spaces
                line = line.lstrip()
                # Replace 2+ spaces with comma
                line = re.sub(r'\s{2,}', ',', line)
                converted_lines.append(line)
            
            # Write back without .txt extension
            filename = f'{dataset_name}_{file_suffix}'
            with open(filename, 'w') as f:
                f.writelines(converted_lines)
        print(f"Converted {dataset_name} files: removed leading spaces, replaced spaces with commas")
        # Remove the original txt files
        os.remove(f'{dataset_name}_TRAIN.txt')
        os.remove(f'{dataset_name}_TEST.txt')
    else:
        !cp {path}/{dataset_name}_TRAIN .
        !cp {path}/{dataset_name}_TEST .
    !mkdir -p {saxdir_name}



#  1 Classification on the FordB dataset

## SAX Only

Prepare the data from the FordB dataset

In [14]:
dataset_name = 'FordB'
saxdir_name = 'saxdirfordb'
prepare_data_sax(dataset_name, saxdir_name)

/home/ayoub/Mr-SEQL/build/Release/bin
Converted FordB files: removed leading spaces, replaced spaces with commas


Run SAX conversion (should last a few seconds max)

We need choose min and max window size. If `max_ws` is too large and the sax conversion takes more than a few seconds, then the classification step may take too long to complete.

In [17]:
max_ws = 100
min_ws = 5
word_length = 6
alphabet = 4

%cd {ROOT_DIR}/build/Release/bin/
!./sax_convert -m 1 -N {max_ws} -n {min_ws} -w {word_length} -a {alphabet} -i {dataset_name}_TRAIN -o {saxdir_name}/sax.train > {saxdir_name}/config

/home/ayoub/Mr-SEQL/build/Release/bin


In [18]:
%cd {ROOT_DIR}/build/Release/bin/
!./sax_convert -m 1 -N {max_ws} -n {min_ws} -w {word_length} -a {alphabet} -i {dataset_name}_TEST -o {saxdir_name}/sax.test

/home/ayoub/Mr-SEQL/build/Release/bin
0 5 6 4
1 10 6 4
2 20 6 4
3 40 6 4
4 80 6 4


CLassify using Mr-SEQL model (2-3 minutes for me)

In [19]:
%cd {ROOT_DIR}/build/Release/bin/
!./mr_seql -t {saxdir_name}/sax.train -T {saxdir_name}/sax.test -o {saxdir_name}

/home/ayoub/Mr-SEQL/build/Release/bin
Number of representations:5
Elapsed Time (TotalLearn,MaxLearn,TotalTest,MaxTest,VectorSpace):87.34466900,28.84382200,3.46137600,1.61272200,22.96864400
(Ensemble) SEQL Accuracy: 0.72098765
TP/FP/TN/FN: 364/181/220/45


Try to train a logistic regression model using the extracted SAX features

In [20]:
%cd {ROOT_DIR}/build/Release/bin/
from src.python.mf_logreg import run_mf_logreg

run_mf_logreg([saxdir_name])

/home/ayoub/Mr-SEQL/build/Release/bin
Trainx source ['saxdirfordb/train.x']
Accuracy with logreg: 0.76296296


We observe that We can improve the accuracy by training a logistic regression model on the SEQL features.

## SFA Only

We now try to use SFA representation on the FordB dataset.

Extract SFA representations (37 min for me)

In [8]:
from src.python.SFA import run_SFA
%cd {ROOT_DIR}/build/Release/bin
%mkdir -p sfadir

min_wl = 100
max_wl = 201

run_SFA('FordB', minwl=min_wl, maxwl=max_wl)

/home/ayoub/Mr-SEQL/build/Release/bin


Loading from: /home/ayoub/Mr-SEQL/build/Release/bin/FordB
/home/ayoub/Mr-SEQL/build/Release/bin/\FordB_TRAIN
Done reading FordB Training Data...  Samples: 3636  Length: 500
Done reading FordB Testing Data...  Samples: 810  Length: 500
Classes: [-1  1]
100
200


Extract SFA features from representation

In [ ]:
%cd {ROOT_DIR}/build/Release/bin/
!./mr_seql -t sfadir/fordb.sfa.train -T sfadir/fordb.sfa.test -o sfadir

Number of representations:2
Elapsed Time (TotalLearn,MaxLearn,TotalTest,MaxTest,VectorSpace):173.12038000,93.68724600,2.94633300,1.79642800,17.12915100
(Ensemble) SEQL Accuracy: 0.71111111
TP/FP/TN/FN: 359/184/217/50


Train a logistic regression model on the SFA features

In [11]:
%cd {ROOT_DIR}/build/Release/bin
run_mf_logreg(["sfadir"])

/home/ayoub/Mr-SEQL/build/Release/bin
Accuracy with logreg: 0.70246914


## Both SAX and SFA

Try to train a logistic regression model using both SAX and SFA features

In [ ]:
%cd {ROOT_DIR}/build/Release/bin
run_mf_logreg([saxdir_name, "sfadir"])

/home/ayoub/Mr-SEQL/build/Release/bin
Accuracy with logreg: 0.79876543


See report for conclusions.

# 2 Classification on the MelbournePedestrian dataset

In [21]:
dataset_name = 'MelbournePedestrian'
saxdir_name = 'saxdirmelbourne'
prepare_data_sax(dataset_name, saxdir_name)

/home/ayoub/Mr-SEQL/build/Release/bin


Converted MelbournePedestrian files: removed leading spaces, replaced spaces with commas


In [23]:
max_ws = 24
min_ws = 1
word_length = 6
alphabet = 8

%cd {ROOT_DIR}/build/Release/bin/
!./sax_convert -m 1 -N {max_ws} -n {min_ws} -w {word_length} -a {alphabet} -i {dataset_name}_TRAIN -o {saxdir_name}/sax.train > {saxdir_name}/config

!./sax_convert -m 1 -N {max_ws} -n {min_ws} -w {word_length} -a {alphabet} -i {dataset_name}_TEST -o {saxdir_name}/sax.test

/home/ayoub/Mr-SEQL/build/Release/bin
0 1 6 8
1 2 6 8
2 4 6 8
3 8 6 8
4 16 6 8


Classify using Mr-SEQL model

In [24]:
%cd {ROOT_DIR}/build/Release/bin/
!./mr_seql -t {saxdir_name}/sax.train -T {saxdir_name}/sax.test -o {saxdir_name}

/home/ayoub/Mr-SEQL/build/Release/bin
Number of representations:5
Number of classes: 10
Elapsed Time (TotalLearn,MaxLearn,TotalTest,MaxTest,VectorSpace):11.73369100,0.70130500,1.75247000,0.11461200,1.57853100
(Ensemble) SEQL Accuracy: 0.72488725


In [ ]:
%cd {ROOT_DIR}/build/Release/bin/
from src.python.mf_logreg import run_mf_logreg

run_mf_logreg([saxdir_name])

/home/ayoub/Mr-SEQL/build/Release/bin
Trainx source ['saxdirmelbourne/train.x.1', 'saxdirmelbourne/train.x.10', 'saxdirmelbourne/train.x.2', 'saxdirmelbourne/train.x.3', 'saxdirmelbourne/train.x.4', 'saxdirmelbourne/train.x.5', 'saxdirmelbourne/train.x.6', 'saxdirmelbourne/train.x.7', 'saxdirmelbourne/train.x.8', 'saxdirmelbourne/train.x.9']
Accuracy with logreg: 0.82451825


In [45]:
%cd {ROOT_DIR}/build/Release/bin/
!./compute_metats -c {saxdir_name}/config -p {saxdir_name}/features -i {dataset_name}_TEST -o {saxdir_name}/test_scores > /dev/null



/home/ayoub/Mr-SEQL/build/Release/bin


In [41]:
%mkdir figures
!python {ROOT_DIR}/src/python/visual_timeseries.py {dataset_name}_TEST {saxdir_name}/test_scores 1 500 1000 1500 2000

Again, see report for conclusions.